# Peak and SWE50 timing components — revised for plotting protocol v5

The two bars preserve the original scientific quantities: **peak-date difference** and **peak-to-50% duration difference**, both Noah-MP minus iSnobal in days. The second is the official Table 3 `Delta_t50_days`; it is **not** the calendar SWE50-date difference. Their sum equals the calendar SWE50-date difference, which is independently reconstructed and saved in the companion CSV.

A positive peak-date difference means Noah-MP peaks later. A positive duration difference means Noah-MP takes longer to reach half of its **own** peak. It does not by itself establish a lower physical melt rate, because peak magnitude and net SWE processes can differ. The first tied maximum and first observed post-peak threshold crossing follow the v0.8.2 daily watershed calculation; no interpolation or smoothing is introduced. The last water year may be a partial available record.

**Run all cells in order** with the v0.8.2 package installed. Table 3 is checked against the daily native watershed SWE tables under the same production root. Missing, duplicate, fractional-duration, out-of-period or inconsistent records stop execution rather than silently disappearing from the chart. The two bars represent timing metrics, not individual models: neutral fills and hatching distinguish them without assigning model colors to differences. No correlation/bias annotation is needed for these timing components.

Keep `East_River_fonts` beside the notebooks or set `FONT_PATHS` explicitly. Nimbus Sans is verified with no fallback. Both 600-dpi PNG and font-embedded PDF are exported with a new `_v5` stem, along with timing data and provenance. Original saved outputs have been cleared.


In [ ]:
from __future__ import annotations
from pathlib import Path
from copy import deepcopy
import math, json, hashlib, warnings
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm
from matplotlib.patches import Patch
from matplotlib.ticker import MultipleLocator
from east_river_workflow.config import load_config, WorkflowConfig


In [ ]:
# Production inputs; all tables, caches and exports use this one output root.
ROOT = Path('/bsuhome/zihaozhang/scratch/iSnobal/East_River_Workflow_Code_v0.8.2')
OUTPUT_DIR = ROOT / 'output' / 'output1'
CONFIG_PATH = ROOT / 'config' / 'east_river_config.yaml'
TABLE3_PATH = OUTPUT_DIR / 'tables' / 'table3_watershed_annual_metrics.csv'
# Optional local font registration; an installed Nimbus Sans also works.
# For Jupyter, set an absolute path here if the working directory differs.
FONT_PATHS = [Path.cwd() / 'East_River_fonts' / 'NimbusSans-Regular.ttf']
MODEL_COLORS = {'iSnobal': '#1F77B4', 'Noah-MP': '#D62728', 'ASO': 'black'}
# Neutral metric categories; blue/red remain reserved for individual model series.
PEAK_COLOR = '0.35'
DEPLETION_COLOR = '0.78'
FIGURE_STEM = 'h2_peak_and_postpeak_swe50_timing_v5'


In [ ]:
def configure_publication_style(font_paths=()):
    for path in font_paths:
        if Path(path).is_file():
            fm.fontManager.addfont(str(path))
    try:
        resolved = fm.findfont(fm.FontProperties(family='Nimbus Sans', weight='normal'),
                               fallback_to_default=False)
    except ValueError as exc:
        raise RuntimeError('Nimbus Sans is required. Install it or set FONT_PATHS to its font file; no fallback is permitted.') from exc
    if fm.FontProperties(fname=resolved).get_name() != 'Nimbus Sans':
        raise RuntimeError(f'Wrong resolved font: {resolved}')
    mpl.rcParams.update({'font.family': 'Nimbus Sans', 'font.size': 18,
        'font.weight': 'normal', 'axes.titlesize': 20, 'axes.titleweight': 'normal',
        'axes.labelweight': 'normal', 'axes.labelsize': 18, 'legend.fontsize': 18,
        'xtick.labelsize': 18, 'ytick.labelsize': 18, 'axes.titlepad': 8,
        'pdf.fonttype': 42, 'ps.fonttype': 42, 'pdf.use14corefonts': False,
        'text.usetex': False, 'axes.grid': False, 'figure.dpi': 120,
        'savefig.dpi': 600, 'lines.linewidth': 2.5})
    print(f'Verified Nimbus Sans: {resolved}')
    return resolved


def validate_table3(frame, years, analysis_start, analysis_end):
    required = {'water_year', 'peak_date_I', 'peak_date_N', 'days_to_50_I',
                'days_to_50_N', 'Delta_t50_days'}
    missing = required - set(frame)
    if missing:
        raise ValueError(f'Table 3 missing columns: {sorted(missing)}')
    years_numeric = np.asarray(years, dtype=float)
    if (not len(years_numeric) or not np.isfinite(years_numeric).all()
            or not np.equal(years_numeric, np.floor(years_numeric)).all()
            or len(set(years_numeric)) != len(years_numeric)):
        raise ValueError('Configured water years must be nonempty, unique integers.')
    years = [int(y) for y in years_numeric]
    frame = frame.copy()
    wy = pd.to_numeric(frame['water_year'], errors='raise')
    if not np.isfinite(wy).all() or not (wy == np.floor(wy)).all():
        raise ValueError('Invalid water_year values in Table 3.')
    frame['water_year'] = wy.astype(int)
    extras = sorted(set(frame['water_year']) - set(years))
    if extras:
        print(f'Table 3 years outside the configured selection: {extras}')
    frame = frame[frame['water_year'].isin(years)].copy()
    if frame['water_year'].duplicated().any():
        raise ValueError('Duplicate Table 3 rows for a selected water year.')
    missing_years = set(years) - set(frame['water_year'])
    if missing_years:
        raise ValueError(f'Table 3 missing water years: {sorted(missing_years)}')
    frame = frame.set_index('water_year').loc[years].reset_index()
    for source in ['I', 'N']:
        col = f'peak_date_{source}'
        dates = pd.to_datetime(frame[col], errors='raise')
        if dates.isna().any() or dates.dt.tz is not None or not dates.eq(dates.dt.normalize()).all():
            raise ValueError(f'{col} must contain nonmissing, timezone-naive daily dates.')
        frame[col] = dates
        duration = pd.to_numeric(frame[f'days_to_50_{source}'], errors='raise')
        if (not np.isfinite(duration).all() or (duration < 0).any()
                or not duration.eq(np.floor(duration)).all()):
            raise ValueError(f'days_to_50_{source} must be finite nonnegative whole days; an unreached SWE50 is undefined, not zero.')
        frame[f'days_to_50_{source}'] = duration.astype(int)
        frame[f'swe50_date_{source}'] = dates + pd.to_timedelta(duration, unit='D')
        for idx, row in frame.iterrows():
            lo = max(pd.Timestamp(int(row.water_year)-1, 10, 1), pd.Timestamp(analysis_start).normalize())
            hi = min(pd.Timestamp(int(row.water_year), 9, 30), pd.Timestamp(analysis_end).normalize())
            if not lo <= row[col] <= row[f'swe50_date_{source}'] <= hi:
                raise ValueError(f'WY {row.water_year}: {source} peak/SWE50 outside the water year or analysis period.')
    official = pd.to_numeric(frame['Delta_t50_days'], errors='raise').to_numpy(float)
    calculated = (frame['days_to_50_N'] - frame['days_to_50_I']).to_numpy(float)
    if not np.isfinite(official).all() or not np.allclose(official, calculated, rtol=0, atol=1e-10):
        raise ValueError('Table 3 Delta_t50_days disagrees with days_to_50_N minus days_to_50_I (or is missing).')
    frame['Delta_t50_days'] = official
    return frame


def validate_daily_timing(table3, table_dir):
    """Check Table 3 against native watershed means, never means of regridded cells."""
    audits = []
    for source, filename in [('I', 'isnobal_watershed_daily.csv'), ('N', 'noahmp_watershed_daily.csv')]:
        path = Path(table_dir) / filename
        daily = pd.read_csv(path)
        if not {'date', 'water_year', 'swe_mm'} <= set(daily):
            raise ValueError(f'Missing date, water_year or swe_mm in {path}')
        daily['date'] = pd.to_datetime(daily['date'], errors='raise')
        if (daily['date'].isna().any() or daily['date'].dt.tz is not None
                or not daily['date'].eq(daily['date'].dt.normalize()).all()
                or daily['date'].duplicated().any()):
            raise ValueError(f'Invalid/duplicate daily dates: {path}')
        expected_wy = daily['date'].dt.year + (daily['date'].dt.month >= 10).astype(int)
        if not pd.to_numeric(daily['water_year'], errors='raise').eq(expected_wy).all():
            raise ValueError(f'Water-year/date disagreement: {path}')
        daily['swe_mm'] = pd.to_numeric(daily['swe_mm'], errors='raise')
        for _, row in table3.iterrows():
            s = daily.loc[expected_wy.eq(row.water_year)].set_index('date')['swe_mm'].sort_index()
            finite = s[np.isfinite(s)]
            if finite.empty or finite.max() <= 0:
                raise ValueError(f'No positive finite peak: {path}, WY {row.water_year}')
            peak_date, peak = finite.idxmax(), float(finite.max())
            post = finite.loc[peak_date:]
            hits = post.index[(post / peak) <= 0.5]
            if not len(hits):
                raise ValueError(f'SWE50 not observed: {path}, WY {row.water_year}')
            swe50 = hits[0]
            if peak_date != row[f'peak_date_{source}'] or swe50 != row[f'swe50_date_{source}']:
                raise ValueError(f'Table 3 and daily watershed SWE disagree: source {source}, WY {row.water_year}')
            peak_col = f'peak_SWE_{source}_mm'
            if peak_col in table3 and not np.isclose(float(row[peak_col]), peak, rtol=1e-9, atol=1e-6):
                raise ValueError(f'Table 3 peak magnitude disagrees: source {source}, WY {row.water_year}')
            expected = pd.date_range(s.index.min(), s.index.max(), freq='D')
            n_missing = int((~np.isfinite(s.reindex(expected))).sum())
            if n_missing:
                warnings.warn(f'{source} WY {row.water_year}: {n_missing} unavailable daily states; crossing is first observed <=50%, with no interpolation.')
            audits.append({'water_year': int(row.water_year), 'source': source,
                'record_start': s.index.min(), 'record_end': s.index.max(),
                'peak_swe_mm': peak, 'peak_date': peak_date, 'swe50_date': swe50,
                'unavailable_daily_states': n_missing, 'table3_agrees': True})
    return pd.DataFrame(audits)


def file_record(path):
    path = Path(path)
    stat = path.stat()
    record = {'path': str(path.resolve()), 'size_bytes': stat.st_size, 'mtime_ns': stat.st_mtime_ns}
    if stat.st_size < 20_000_000:
        record['sha256'] = hashlib.sha256(path.read_bytes()).hexdigest()
    return record


def save_publication_figure(fig, directory, stem):
    # Check actual text objects too: rcParams alone cannot prevent explicit overrides.
    for item in fig.findobj(mpl.text.Text):
        if item.get_text():
            resolved = fm.findfont(item.get_fontproperties(), fallback_to_default=False)
            if fm.FontProperties(fname=resolved).get_name() != 'Nimbus Sans':
                raise RuntimeError(f'Figure text uses a different font: {item.get_text()}')
    directory = Path(directory)
    directory.mkdir(parents=True, exist_ok=True)
    for extension in ['png', 'pdf']:
        path = directory / f'{stem}.{extension}'
        # PDF rasterized dense scatter is also 600 dpi; axes/text remain vector.
        fig.savefig(path, dpi=600, bbox_inches='tight', pad_inches=0.05, facecolor='white')
        print(f'Saved: {path}')


In [ ]:
cfg_original = load_config(CONFIG_PATH)
data = deepcopy(cfg_original.data)
if cfg_original.output_dir.resolve() != OUTPUT_DIR.resolve():
    print(f'Config output_dir is {cfg_original.output_dir}; using the user-specified production root: {OUTPUT_DIR}')
data['paths']['output_dir'] = str(OUTPUT_DIR)
cfg = WorkflowConfig(data=data, source_path=cfg_original.source_path)
years = cfg.section('project')['water_years']
table3 = validate_table3(pd.read_csv(TABLE3_PATH), years, cfg.analysis_start, cfg.analysis_end)
years = table3['water_year'].tolist()
timing_audit = validate_daily_timing(table3, OUTPUT_DIR / 'tables')
font_file = configure_publication_style(FONT_PATHS)
print(timing_audit.to_string(index=False))
print('Peak and SWE50 refer to the available analysis record; the final WY may be partial.')
input_records = [file_record(p) for p in [CONFIG_PATH, TABLE3_PATH,
    OUTPUT_DIR/'tables/isnobal_watershed_daily.csv', OUTPUT_DIR/'tables/noahmp_watershed_daily.csv']]


In [ ]:
df = table3.copy()
# Both differences are Noah-MP minus iSnobal. These are different metrics.
df['delta_peak_date_days'] = (df['peak_date_N'] - df['peak_date_I']).dt.days
df['delta_peak_to_50_days'] = df['days_to_50_N'] - df['days_to_50_I']
df['delta_calendar_swe50_days'] = (df['swe50_date_N'] - df['swe50_date_I']).dt.days
if not df['delta_calendar_swe50_days'].eq(df['delta_peak_date_days'] + df['delta_peak_to_50_days']).all():
    raise ValueError('Calendar SWE50 timing identity failed.')
print(df[['water_year', 'delta_peak_date_days', 'delta_peak_to_50_days', 'delta_calendar_swe50_days']].to_string(index=False))


In [ ]:
fig, ax = plt.subplots(figsize=(18, 8.5))

x = np.arange(len(df), dtype=float)
bar_width = 0.34


# Peak-date contribution
bars_peak = ax.bar(
    x - bar_width / 2,
    df["delta_peak_date_days"],
    width=bar_width,
    color=PEAK_COLOR,
    edgecolor="black",
    linewidth=1.5,
    zorder=2,
)


# Post-peak depletion contribution
bars_depletion = ax.bar(
    x + bar_width / 2,
    df["delta_peak_to_50_days"],
    width=bar_width,
    color=DEPLETION_COLOR,
    hatch="///",
    edgecolor="black",
    linewidth=1.5,
    zorder=2,
)


# Zero reference line
ax.axhline(
    0.0,
    color="0.35",
    lw=1.0,
    zorder=1,
)


# -------------------------------------------------------------------------
# VALUE LABELS
# -------------------------------------------------------------------------
def add_bar_labels(
    ax,
    bars,
    values,
    offset=0.8,
):
    for bar, value in zip(bars, values):

        if not np.isfinite(value):
            continue

        x_center = (
            bar.get_x()
            + bar.get_width() / 2
        )

        if value > 0:
            y = value + offset
            va = "bottom"

        elif value < 0:
            y = value - offset
            va = "top"

        else:
            y = value + offset
            va = "bottom"

        label = (
            f"+{value:.0f}"
            if value > 0
            else f"{value:.0f}"
        )

        ax.text(
            x_center,
            y,
            label,
            ha="center",
            va=va,
            fontsize=18,
        )


add_bar_labels(
    ax,
    bars_peak,
    df["delta_peak_date_days"].to_numpy(dtype=float),
)

add_bar_labels(
    ax,
    bars_depletion,
    df["delta_peak_to_50_days"].to_numpy(dtype=float),
)


# -------------------------------------------------------------------------
# X / Y AXES
# -------------------------------------------------------------------------
ax.set_xticks(x)

# Same style as your supplied grouped-bar notebook:
# use only the numeric water year, not "WY 2018".
ax.set_xticklabels(
    [
        f"{int(wy)}"
        for wy in df["water_year"]
    ]
)

ax.set_xlabel("Water year")

ax.set_ylabel("Timing difference (days)\nNoah-MP minus iSnobal")


# No main figure title.


# -------------------------------------------------------------------------
# CLEAN DYNAMIC Y LIMITS
# -------------------------------------------------------------------------
all_values = df[['delta_peak_date_days', 'delta_peak_to_50_days']].to_numpy(float)
bound = max(10.0, math.ceil((np.abs(all_values).max() * 1.35 + 3) / 5) * 5)
ax.set_ylim(-bound, bound)
ax.yaxis.set_major_locator(MultipleLocator(max(5, math.ceil(bound / 30) * 5)))
ax.set_facecolor('0.94')
ax.yaxis.grid(True, linestyle='--', linewidth=0.8, alpha=0.45)
ax.xaxis.grid(False)
ax.set_axisbelow(True)

# -------------------------------------------------------------------------
# AXIS STYLING
# -------------------------------------------------------------------------
ax.spines[
    ["top", "right"]
].set_visible(False)

ax.spines[
    "left"
].set_linewidth(1.0)

ax.spines[
    "bottom"
].set_linewidth(1.0)

ax.tick_params(
    axis="both",
    which="major",
    width=1.0,
    length=5,
)


# -------------------------------------------------------------------------
# LEGEND
# -------------------------------------------------------------------------
legend_handles = [

    Patch(
        facecolor=PEAK_COLOR,
        edgecolor="black",
        linewidth=1.5,
        label="Peak-date difference",
    ),

    Patch(
        facecolor=DEPLETION_COLOR,
    hatch="///",
        edgecolor="black",
        linewidth=1.5,
        label="Peak-to-50% duration difference",
    ),
]


ax.legend(
    handles=legend_handles,
    loc="upper left",
    ncol=1,
    frameon=True,
    facecolor="white",
    edgecolor="0.4",
    framealpha=0.9,
    fancybox=False,
    borderpad=0.5,
    handlelength=2.2,
)


# -------------------------------------------------------------------------
# SPACING
# -------------------------------------------------------------------------
fig.subplots_adjust(
    top=0.97,
    bottom=0.15,
    left=0.11,
    right=0.98,
)


In [ ]:
save_publication_figure(fig, OUTPUT_DIR/'figures/scientific', FIGURE_STEM)
plt.show()
columns = ['water_year', 'peak_date_I', 'peak_date_N', 'swe50_date_I', 'swe50_date_N',
    'days_to_50_I', 'days_to_50_N', 'Delta_t50_days', 'delta_peak_date_days',
    'delta_peak_to_50_days', 'delta_calendar_swe50_days']
table_dir = OUTPUT_DIR/'tables'
table_dir.mkdir(parents=True, exist_ok=True)
df[columns].to_csv(table_dir/f'{FIGURE_STEM}.csv', index=False)
timing_audit.to_csv(table_dir/f'{FIGURE_STEM}_audit.csv', index=False)
(table_dir/f'{FIGURE_STEM}_provenance.json').write_text(json.dumps({
    'sign': 'Noah-MP minus iSnobal', 'font': file_record(font_file), 'inputs': input_records,
    'duration': 'own peak to first observed daily SWE <= 50% of own peak',
    'calendar_identity': 'calendar SWE50 difference = peak date difference + duration difference'}, indent=2), encoding='utf-8')
